# Phase 4 — Feature engineering justifié par ablation (XGBoost, FD001)

Applique la **section 7** de [`docs/protocole_evaluation.md`](../docs/protocole_evaluation.md) au XGBoost de la Phase 3.

## Point de départ, dit franchement

Les 46 features du notebook 03 sont : 15 capteurs lissés (moyenne mobile 5 cycles, Phase 1) + leur moyenne glissante sur 5 cycles + leur écart-type glissant sur 5 cycles + `time_cycles`.

- Aucun document du dépôt ne justifie le choix « moyenne + écart-type sur 5 cycles ». Le cahier des charges (Phase 3) demande moyenne, variance, asymétrie et kurtosis ; seules deux de ces statistiques avaient été reprises, **sans hypothèse écrite**.
- L'ablation de la Phase 3 (§5) a montré que ces colonnes n'apportent rien de mesurable (écarts de 0,2 de RMSE contre un bruit d'environ 1,3 entre plis), que tout le gain vient de `time_cycles`, et que **16 features (15 capteurs + `time_cycles`) font aussi bien que 46**.
- Une raison probable : la « moyenne glissante » est la moyenne sur 5 cycles d'un capteur **déjà** lissé sur 5 cycles (moyenne de moyennes, corrélation 0,987 mesurée en Phase 3). Elle décrit le même niveau, avec un peu de retard.

**Référence à battre : les 16 features** (`ref16`). Les 46 actuelles (`ref46`) sont aussi mesurées.

## Ce que ce notebook fait

Pour chaque famille de features : une hypothèse écrite **avant** le test → un test d'ablation sur les mêmes plis par moteur que le notebook 03 → une conclusion chiffrée, y compris quand la famille n'apporte rien. Puis sélection (règle écrite avant les résultats), importance des features (gain **et** permutation), et **un seul** passage sur le test officiel pour le jeu retenu.

## Contraintes respectées

- features **causales** (la ligne du cycle t n'utilise que les cycles ≤ t), **moteur par moteur** — vérifié par un test automatique en section 2 ;
- mêmes réglages XGBoost que le notebook 03 (`n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42`) ;
- mêmes plis : `GroupKFold(n_splits=5, shuffle=True, random_state=42)` (graine 42). Pour mesurer le bruit, la validation est **répétée avec deux autres graines de plis** (43 et 44) ; la graine 42 reste la référence comparable au notebook 03 ;
- jamais le test officiel pour choisir quoi que ce soit.

## 1. Chargement et préparation (identique au notebook 03)

On repart de `src/preprocessing.py` sans le modifier. Deux versions du signal sont gardées :
- le signal **normalisé non lissé** (nécessaire pour la moyenne mobile exponentielle, qui est une alternative au lissage) ;
- le signal **lissé** par la moyenne mobile 5 cycles (Phase 1), base des 16 features de référence.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5          # fenetre du lissage de la Phase 1
PLAFOND_RUL = 125    # plafond de la RUL, Phase 2
GRAINES_PLIS = [42, 43, 44]  # 42 = plis du notebook 03 ; 43 et 44 servent a mesurer le bruit du decoupage
# memes reglages XGBoost que le notebook 03, pour que seule la liste de features change
REGLAGES_XGB = dict(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)

In [ ]:
train_brut = load_fd001("train")  # fichier d'entrainement FD001

sensor_cols = get_sensor_columns(train_brut.columns)  # les 21 capteurs bruts
capteurs_constants = find_constant_sensors(train_brut, sensor_cols)  # 6 capteurs constants (Phase 1)
train_brut = train_brut.drop(columns=capteurs_constants)  # on les retire
capteurs = get_sensor_columns(train_brut.columns)  # les 15 capteurs informatifs

# normalisation min-max (le scaler est garde pour le test) : ici le signal n'est PAS encore lisse
train_norm, scaler = normalize_minmax(train_brut, capteurs)
train_norm = calculer_rul(train_norm)  # RUL brute par cycle
train_norm = plafonner_rul(train_norm, plafond=PLAFOND_RUL)  # RUL plafonnee a 125

# les features par fenetre supposent des lignes triees par moteur puis par cycle (ordre du fichier)
ordre_attendu = train_norm.sort_values(["unit_number", "time_cycles"]).index
assert (ordre_attendu == train_norm.index).all(), "le tableau n'est pas trie par moteur puis par cycle"

y = train_norm["RUL_cappee"]      # cible (Phase 2)
groups = train_norm["unit_number"]  # sert uniquement a GroupKFold (quel moteur)
print(f"train : {train_norm.shape}, {groups.nunique()} moteurs, {len(capteurs)} capteurs")

## 2. Fabrication des features candidates

Une seule fonction calcule **toutes** les familles candidates (pour le train comme pour le test, avec exactement le même code) ; chaque test d'ablation n'en sélectionne ensuite que quelques colonnes. Chaque colonne s'appelle `capteur|statistique_k` (`k` = taille de la fenêtre en cycles).

| Statistique | Définition (cycle t, fenêtre k) | Calculée sur |
|---|---|---|
| `moy_k` | moyenne des k derniers cycles | capteur lissé |
| `std_k`, `var_k` | écart-type / variance des k derniers cycles | capteur lissé |
| `skew_k`, `kurt_k` | asymétrie / aplatissement des k derniers cycles | capteur lissé |
| `pente_k` | pente de la droite ajustée (moindres carrés) sur les k derniers cycles | capteur lissé |
| `ewma_k` | moyenne mobile exponentielle, `span=k` | capteur **brut** normalisé |
| `ecartinit_k` | valeur lissée actuelle − moyenne de ses `min(t, k)` premiers cycles | capteur lissé |

Valeurs indéfinies (début de vie : écart-type avec 1 point, asymétrie avec moins de 3 points, kurtosis avec moins de 4, pente avec 1 point) remplacées par 0, comme le notebook 03 le fait déjà pour l'écart-type.

`ecartinit_k` est rendu causal : au cycle t < k, la référence est la moyenne des t premiers cycles seulement (pas des k premiers, qui incluraient le futur).

In [ ]:
# score NASA (Saxena et al., 2008), version vectorisee, en MOYENNE par prediction (protocole, section 1.5)
def score_nasa_moyen(y_vrai, y_pred):
    """
    Score NASA moyen par prediction : exp(-d/13)-1 si d<0 (sous-estimation), exp(d/10)-1 sinon (surestimation),
    avec d = y_pred - y_vrai. Meme formule que score_nasa du notebook 03, divisee par le nombre de predictions.
    """
    d = np.asarray(y_pred, dtype=float) - np.asarray(y_vrai, dtype=float)  # ecart signe
    penalites = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)    # penalite de chaque exemple
    return float(np.mean(penalites))


def score_nasa_boucle(y_true, y_pred):
    """Copie de score_nasa du notebook 03 (boucle explicite, SOMME), uniquement pour verifier la version vectorisee."""
    total = 0.0
    for vrai, predit in zip(y_true, y_pred):
        d = predit - vrai
        if d < 0:
            total += np.exp(-d / 13) - 1
        else:
            total += np.exp(d / 10) - 1
    return total


generateur = np.random.default_rng(0)  # graine fixe : le test est reproductible
vrais = generateur.uniform(0, 125, 500)
predits = vrais + generateur.normal(0, 15, 500)
assert abs(score_nasa_moyen(vrais, predits) * 500 - score_nasa_boucle(vrais, predits)) < 1e-6
print("Score NASA vectorise = score NASA du notebook 03 (somme / nombre de predictions) : OK")

In [ ]:
# fenetres testees pour chaque statistique (hypotheses en section 4)
FENETRES = {
    "moy": [5, 10, 20, 30],
    "std": [5, 10, 20, 30],
    "var": [5, 20],
    "skew": [5, 20],
    "kurt": [5, 20],
    "pente": [5, 10, 20, 30],
    "ewma": [5, 10, 20],
    "ecartinit": [5, 10, 20],
}


def pente_moindres_carres(valeurs):
    """
    Pente de la droite de regression lineaire de `valeurs` en fonction de leur rang (0, 1, 2, ...).

    Parameters
    ----------
    valeurs : numpy.ndarray
        Les valeurs d'un capteur sur la fenetre (au moins 2 points).

    Returns
    -------
    float
        Variation moyenne du capteur par cycle sur la fenetre.
    """
    rangs = np.arange(len(valeurs), dtype=float)       # 0, 1, 2, ... : le temps, en cycles
    rangs_centres = rangs - rangs.mean()               # on centre pour simplifier la formule
    # pente = covariance(rang, valeur) / variance(rang), ecrite avec des sommes
    numerateur = np.sum(rangs_centres * (valeurs - valeurs.mean()))
    denominateur = np.sum(rangs_centres ** 2)
    return numerateur / denominateur


def statistique_fenetre(serie_lisse, serie_brute, statistique, k):
    """
    Calcule UNE statistique causale (cycle t -> cycles <= t) sur UN moteur et UN capteur.

    Parameters
    ----------
    serie_lisse : pandas.Series
        Signal lisse du capteur pour ce moteur, dans l'ordre des cycles.
    serie_brute : pandas.Series
        Signal normalise NON lisse du meme capteur et du meme moteur (pour l'EWMA).
    statistique : str
        "moy", "std", "var", "skew", "kurt", "pente", "ewma" ou "ecartinit".
    k : int
        Taille de la fenetre (ou span pour l'EWMA, nombre de cycles de reference pour ecartinit).

    Returns
    -------
    numpy.ndarray
        Une valeur par cycle ; les valeurs indefinies (debut de vie) sont remplacees par 0.
    """
    if statistique == "moy":
        resultat = serie_lisse.rolling(k, min_periods=1).mean()
    elif statistique == "std":
        resultat = serie_lisse.rolling(k, min_periods=1).std()
    elif statistique == "var":
        resultat = serie_lisse.rolling(k, min_periods=1).var()
    elif statistique == "skew":
        resultat = serie_lisse.rolling(k, min_periods=3).skew()   # il faut au moins 3 points
    elif statistique == "kurt":
        resultat = serie_lisse.rolling(k, min_periods=4).kurt()   # il faut au moins 4 points
    elif statistique == "pente":
        resultat = serie_lisse.rolling(k, min_periods=2).apply(pente_moindres_carres, raw=True)
    elif statistique == "ewma":
        # adjust=False : formule recursive classique, s_t = alpha * x_t + (1 - alpha) * s_{t-1}, causale
        resultat = serie_brute.ewm(span=k, adjust=False).mean()
    elif statistique == "ecartinit":
        valeurs = serie_lisse.values
        somme_cumulee = np.cumsum(valeurs)                     # somme des t+1 premiers cycles
        nb_cycles = np.arange(1, len(valeurs) + 1)             # 1, 2, 3, ... : combien de cycles connus a l'instant t
        moyenne_debut = somme_cumulee / nb_cycles              # moyenne des cycles 1..t (cas t < k)
        if len(valeurs) >= k:
            moyenne_debut[k:] = somme_cumulee[k - 1] / k       # t >= k : moyenne FIGEE des k premiers cycles
        resultat = pd.Series(valeurs - moyenne_debut)
    else:
        raise ValueError(f"statistique inconnue : {statistique}")

    return resultat.fillna(0.0).values  # debut de vie : statistique indefinie -> 0


def calculer_features_candidates(df_lisse, df_norm, capteurs):
    """
    Calcule toutes les features candidates, moteur par moteur, capteur par capteur.

    Parameters
    ----------
    df_lisse : pandas.DataFrame
        Tableau avec les capteurs LISSES (sortie de lisser_capteurs), trie par moteur puis cycle.
    df_norm : pandas.DataFrame
        Meme tableau avec les capteurs normalises NON lisses, memes lignes dans le meme ordre.
    capteurs : list de str
        Noms des capteurs.

    Returns
    -------
    pandas.DataFrame
        Une colonne "capteur|statistique_k" par combinaison, meme index que df_lisse.
    """
    nb_lignes = len(df_lisse)
    unites = df_lisse["unit_number"].values
    colonnes = {}  # nom de colonne -> tableau numpy rempli moteur apres moteur

    for capteur in capteurs:  # un capteur a la fois
        for statistique, liste_k in FENETRES.items():
            for k in liste_k:
                colonnes[f"{capteur}|{statistique}_{k}"] = np.zeros(nb_lignes)  # espace reserve

    for moteur_id in np.unique(unites):  # un moteur a la fois : jamais de melange entre deux moteurs
        positions = np.where(unites == moteur_id)[0]  # numeros de ligne de ce moteur
        for capteur in capteurs:
            serie_lisse = pd.Series(df_lisse[capteur].values[positions])  # signal lisse de ce moteur
            serie_brute = pd.Series(df_norm[capteur].values[positions])   # signal brut de ce moteur
            for statistique, liste_k in FENETRES.items():
                for k in liste_k:
                    valeurs = statistique_fenetre(serie_lisse, serie_brute, statistique, k)
                    colonnes[f"{capteur}|{statistique}_{k}"][positions] = valeurs

    return pd.DataFrame(colonnes, index=df_lisse.index)


def preparer_features(df_norm, capteurs):
    """
    Pipeline complet : lissage (Phase 1) puis toutes les features candidates.

    Parameters
    ----------
    df_norm : pandas.DataFrame
        Tableau normalise, NON lisse, trie par moteur puis cycle.
    capteurs : list de str
        Noms des capteurs.

    Returns
    -------
    pandas.DataFrame
        Colonnes : capteurs lisses, time_cycles, toutes les features candidates.
    """
    df_lisse = lisser_capteurs(df_norm, capteurs, fenetre=FENETRE)  # lissage moteur par moteur (src/)
    candidates = calculer_features_candidates(df_lisse, df_norm, capteurs)
    base = df_lisse[capteurs + ["time_cycles"]]  # 15 capteurs lisses + numero de cycle
    return pd.concat([base, candidates], axis=1)

In [ ]:
# --- Test 1 : causalite et independance des moteurs ---
# On calcule les features sur 3 moteurs, puis sur les memes 3 moteurs dont le moteur 1 est COUPE apres le
# cycle 100. Si une feature utilisait le futur ou melangeait deux moteurs, les lignes communes differeraient.
petit = train_norm[train_norm["unit_number"] <= 3].reset_index(drop=True)
complet = preparer_features(petit, capteurs)

masque_garde = (petit["unit_number"] != 1) | (petit["time_cycles"] <= 100)  # lignes qui existent encore apres la coupe
petit_coupe = petit[masque_garde].reset_index(drop=True)
coupe = preparer_features(petit_coupe, capteurs)

complet_restreint = complet[masque_garde.values].reset_index(drop=True)  # memes lignes, calculees AVEC le futur
assert complet_restreint.shape == coupe.shape
assert np.allclose(complet_restreint.values, coupe.values, atol=1e-9), "une feature depend du futur ou d'un autre moteur !"
print(f"Test causalite / independance des moteurs : OK ({coupe.shape[1]} colonnes comparees sur {len(coupe)} lignes)")

In [ ]:
# --- Test 2 : nos colonnes moy_5 / std_5 sont bien celles du notebook 03 ---
def features_notebook_03(df_lisse_petit, capteurs, fenetre):
    """Copie (boucles explicites) de ajouter_features_glissantes du notebook 03, pour comparaison."""
    df = df_lisse_petit.copy()
    for capteur in capteurs:
        df[capteur + "_moy"] = 0.0
        df[capteur + "_std"] = 0.0
        for moteur_id in df["unit_number"].unique():
            masque = df["unit_number"] == moteur_id
            df.loc[masque, capteur + "_moy"] = df.loc[masque, capteur].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, capteur + "_std"] = df.loc[masque, capteur].rolling(fenetre, min_periods=1).std()
    return df.fillna(0.0)  # 1er cycle : ecart-type indefini -> 0 (comme le notebook 03)

petit_lisse = lisser_capteurs(petit, capteurs, fenetre=FENETRE)
ancien = features_notebook_03(petit_lisse, capteurs, FENETRE)
for capteur in capteurs:
    assert np.allclose(ancien[capteur + "_moy"].values, complet[f"{capteur}|moy_5"].values, atol=1e-12)
    assert np.allclose(ancien[capteur + "_std"].values, complet[f"{capteur}|std_5"].values, atol=1e-12)
print("moy_5 et std_5 = features du notebook 03 : OK")

In [ ]:
# --- Calcul sur tout le train (le plus long du notebook : on mesure le temps) ---
debut = time.time()
X_train = preparer_features(train_norm, capteurs)
print(f"Features calculees en {time.time() - debut:.0f} s : {X_train.shape[1]} colonnes, {len(X_train)} lignes")
assert X_train.isna().sum().sum() == 0, "il reste des NaN"

# apercu sur le moteur 1, capteur 11 : une feature de chaque famille (pour verifier que les valeurs ont du sens)
colonnes_apercu = ["time_cycles", "sensor_11", "sensor_11|std_5", "sensor_11|pente_10", "sensor_11|ewma_10", "sensor_11|ecartinit_10"]
X_train.loc[train_norm["unit_number"] == 1, colonnes_apercu].iloc[[0, 1, 2, 9, 49, 99, 150, 191]]

## 3. Cadre d'évaluation et règle de décision (écrits avant de voir les résultats)

**Plis.** Les plis par moteur sont calculés une fois par graine (42, 43, 44) et réutilisés par tous les jeux de features. Un `assert` vérifie qu'aucun moteur n'est des deux côtés. Un jeu de features est donc évalué sur 3 × 5 = 15 couples (graine, pli) ; les 5 de la graine 42 sont ceux du notebook 03.

**Métriques** (par pli, sur toutes les lignes de validation, cible plafonnée à 125) : RMSE (métrique principale), MAE, score NASA **moyen par prédiction**, et RMSE restreint à la fin de vie (RUL vraie ≤ 30), seule zone où une décision de maintenance se prend.

**Comparaison.** Chaque jeu est comparé à `ref16` **pli par pli** (mêmes plis → différences appariées), ce qui est plus fin que de comparer deux moyennes avec leur écart-type : le bruit du découpage est largement commun aux deux jeux et s'annule dans la différence.

**Règle de décision.** Un jeu candidat est *retenu* seulement si les trois conditions sont vraies :
1. gain moyen de RMSE vs `ref16` ≥ **0,30 cycle** (environ le quart du bruit entre plis, 1,2, mesuré en Phase 3) ;
2. il bat `ref16` sur au moins **12 des 15** couples (graine, pli) ;
3. son score NASA moyen n'est pas dégradé de plus de **5 %**.

Parmi les jeux retenus, on garde celui au meilleur RMSE ; si un jeu avec **moins de features** est à moins de 0,10 cycle du meilleur, on prend le plus simple (règle 9 d'`AGENTS.md`). Si rien n'est retenu, le jeu final est `ref16`.

Un test de Wilcoxon apparié est rapporté **à titre indicatif** : les 15 couples ne sont pas indépendants (les plis d'une même graine partagent des moteurs, et les 3 graines réutilisent les mêmes 100 moteurs), donc la p-valeur est optimiste. Avec 5 plis seulement, la plus petite p-valeur possible vaut 0,0625.

In [ ]:
# plis par moteur, un jeu par graine ; la graine 42 est celle du notebook 03
plis_par_graine = {}
for graine in GRAINES_PLIS:
    gkf = GroupKFold(n_splits=5, shuffle=True, random_state=graine)
    plis = list(gkf.split(X_train, y, groups=groups))  # calcule UNE fois, reutilise pour tous les jeux
    for idx_train, idx_val in plis:  # verification explicite : aucun moteur des deux cotes
        moteurs_train = set(groups.iloc[idx_train])
        moteurs_val = set(groups.iloc[idx_val])
        assert len(moteurs_train & moteurs_val) == 0, "un moteur est present des deux cotes !"
    plis_par_graine[graine] = plis
print(f"Plis calcules pour les graines {GRAINES_PLIS} ({len(plis_par_graine[42])} plis chacune), moteurs jamais partages.")


def evaluer_jeu(colonnes):
    """
    Evalue un jeu de features par validation croisee par moteur, sur toutes les graines de plis.

    Parameters
    ----------
    colonnes : list de str
        Les colonnes de X_train a utiliser comme features.

    Returns
    -------
    pandas.DataFrame
        Une ligne par (graine, pli) avec rmse, mae, nasa_moyen, rmse_fin (RUL vraie <= 30).
    """
    X_jeu = X_train[colonnes]  # seulement les colonnes de ce jeu
    lignes = []
    for graine, plis in plis_par_graine.items():  # les 3 graines de plis
        for numero_pli, (idx_train, idx_val) in enumerate(plis):  # les 5 plis de la graine
            modele = XGBRegressor(**REGLAGES_XGB)  # modele tout neuf, memes reglages que le notebook 03
            modele.fit(X_jeu.iloc[idx_train], y.iloc[idx_train])
            y_pred = modele.predict(X_jeu.iloc[idx_val])
            y_val = y.iloc[idx_val].values

            erreur = y_pred - y_val                          # positif = surestimation
            masque_fin = y_val <= 30                         # fin de vie : RUL vraie <= 30 cycles
            lignes.append({
                "graine": graine,
                "pli": numero_pli + 1,
                "rmse": float(np.sqrt(np.mean(erreur ** 2))),
                "mae": float(np.mean(np.abs(erreur))),
                "nasa_moyen": score_nasa_moyen(y_val, y_pred),
                "rmse_fin": float(np.sqrt(np.mean(erreur[masque_fin] ** 2))),
            })
    return pd.DataFrame(lignes)

## 4. Hypothèses écrites avant les tests

Toutes les familles sont testées **ajoutées aux 16 features de référence** (`ref16` = 15 capteurs lissés + `time_cycles`), une famille à la fois. Les familles de `ref46` (moy_5, std_5) servent de point de comparaison.

| Famille | Hypothèse | Ce que j'attends |
|---|---|---|
| **F1** variance / asymétrie / kurtosis | Le signal devient plus dispersé en fin de vie (Phase 2 : écart-type ×2 à ×2,7 pour RUL ≤ 125). La variance devrait aider surtout en fin de vie (`rmse_fin`). L'asymétrie et le kurtosis sur **5 points** sont des estimateurs très bruités, donc je m'attends à rien en k=5 et, au mieux, un petit effet en k=20. | Variance : gain faible possible. Skew/kurt : rien. |
| **F2** pente | Le niveau d'un capteur dit *où on en est* ; la pente dit *à quelle vitesse ça se dégrade*, information que le niveau seul ne contient pas. Fenêtre trop courte (5) = pente bruitée, donc 10 à 20 devraient être meilleures. | **Famille la plus plausible** d'un vrai gain. |
| **F3** tailles de fenêtre | Une moyenne sur 10-30 cycles d'un signal déjà lissé est une copie retardée du niveau : redondante. Un écart-type sur fenêtre plus longue est une estimation de dispersion plus stable que sur 5 points. | Moyennes : rien. Écarts-types longs : léger gain possible. |
| **F4** EWMA | Autre façon de lisser, calculée sur le signal brut. En **ajout**, elle est redondante avec le capteur lissé. En **remplacement** du lissage (même nombre de features que `ref16`), elle devrait faire jeu égal : un autre filtre, pas une autre information. | Ajout : rien. Remplacement : égalité. |
| **F5** écart à l'état initial | Les moteurs ne partent pas du même niveau (différences de fabrication). Soustraire le niveau de départ isole la dérive propre à la dégradation, et devrait aider le modèle à comparer des moteurs différents. | Gain plausible, deuxième après la pente. |

Ces hypothèses sont des paris : le but de l'ablation est précisément de les mettre à l'épreuve, y compris contre mon intuition.

In [ ]:
# ---- définition de tous les jeux testés : nom -> (famille, liste de colonnes) ----
def colonnes_de(statistique, k):
    """Liste des 15 colonnes 'capteur|statistique_k' (une par capteur)."""
    liste = []
    for capteur in capteurs:
        liste.append(f"{capteur}|{statistique}_{k}")
    return liste


base16 = capteurs + ["time_cycles"]  # reference : 15 capteurs lisses + numero de cycle

configs = {}  # on garde l'ordre d'insertion
configs["ref16 : capteurs + time_cycles"] = {"famille": "ref", "colonnes": base16}
configs["ref46 : + moy_5 + std_5 (actuel)"] = {"famille": "ref", "colonnes": base16 + colonnes_de("moy", 5) + colonnes_de("std", 5)}

# F1 : variance, asymetrie, kurtosis
for k in [5, 20]:
    for statistique in ["var", "skew", "kurt"]:
        configs[f"F1 +{statistique}_{k}"] = {"famille": "F1", "colonnes": base16 + colonnes_de(statistique, k)}
    configs[f"F1 +var+skew+kurt_{k}"] = {
        "famille": "F1",
        "colonnes": base16 + colonnes_de("var", k) + colonnes_de("skew", k) + colonnes_de("kurt", k),
    }

# F2 : pente
for k in [5, 10, 20, 30]:
    configs[f"F2 +pente_{k}"] = {"famille": "F2", "colonnes": base16 + colonnes_de("pente", k)}

# F3 : tailles de fenetre (ecart-type et moyenne)
for k in [5, 10, 20, 30]:
    configs[f"F3 +std_{k}"] = {"famille": "F3", "colonnes": base16 + colonnes_de("std", k)}
for k in [5, 10, 20, 30]:
    configs[f"F3 +moy_{k}"] = {"famille": "F3", "colonnes": base16 + colonnes_de("moy", k)}

# F4 : EWMA, en ajout puis en remplacement du lissage (les 15 capteurs lisses sont alors remplaces)
for k in [5, 10, 20]:
    configs[f"F4 +ewma_{k}"] = {"famille": "F4", "colonnes": base16 + colonnes_de("ewma", k)}
for k in [5, 10, 20]:
    configs[f"F4 ewma_{k} remplace le lissage"] = {"famille": "F4", "colonnes": colonnes_de("ewma", k) + ["time_cycles"]}

# F5 : ecart a l'etat initial
for k in [5, 10, 20]:
    configs[f"F5 +ecartinit_{k}"] = {"famille": "F5", "colonnes": base16 + colonnes_de("ecartinit", k)}

for nom, config in configs.items():  # verification : toutes les colonnes existent
    for colonne in config["colonnes"]:
        assert colonne in X_train.columns, f"colonne absente : {colonne}"
print(f"{len(configs)} jeux de features a evaluer, soit {len(configs) * 15} entrainements XGBoost.")

In [ ]:
# ---- sanity check avant le gros calcul : ref46 doit redonner ~16.86 (graine 42), ref16 ~16.95 (notebook 03 section 5) ----
resultats = {}
for nom in ["ref16 : capteurs + time_cycles", "ref46 : + moy_5 + std_5 (actuel)"]:
    debut = time.time()
    resultats[nom] = evaluer_jeu(configs[nom]["colonnes"])
    rmse_42 = resultats[nom][resultats[nom]["graine"] == 42]["rmse"].mean()
    print(f"{nom:36s} RMSE moyen (graine 42) = {rmse_42:.2f}   [{time.time() - debut:.0f} s]")

rmse_ref46_42 = resultats["ref46 : + moy_5 + std_5 (actuel)"].query("graine == 42")["rmse"].mean()
rmse_ref16_42 = resultats["ref16 : capteurs + time_cycles"].query("graine == 42")["rmse"].mean()
# tolerance 0.15 : XGBoost peut varier legerement d'un environnement a l'autre (note de reproductibilite, doc 03)
assert abs(rmse_ref46_42 - 16.86) < 0.15, f"ref46 devrait redonner 16.86, obtenu {rmse_ref46_42:.2f}"
assert abs(rmse_ref16_42 - 16.95) < 0.15, f"ref16 devrait redonner 16.95, obtenu {rmse_ref16_42:.2f}"
print("Les chiffres du notebook 03 sont retrouves.")

In [ ]:
# ---- le gros calcul : tous les jeux, memes plis, memes reglages ----
debut_total = time.time()
for nom, config in configs.items():
    if nom in resultats:  # deja calcule (les deux references)
        continue
    debut = time.time()
    resultats[nom] = evaluer_jeu(config["colonnes"])
    rmse_moyen = resultats[nom]["rmse"].mean()
    print(f"{nom:40s} {len(config['colonnes']):3d} features   RMSE moyen (15 plis) = {rmse_moyen:.2f}   [{time.time() - debut:.0f} s]")
print(f"\nTemps total : {(time.time() - debut_total) / 60:.1f} min")

## 5. Tableau d'ablation

Colonnes :
- `RMSE 42` : RMSE moyen ± écart-type entre les 5 plis de la graine 42 (les plis du notebook 03) ;
- `RMSE 15` : RMSE moyen sur les 15 couples (graine, pli) ;
- `gain` : RMSE(`ref16`) − RMSE(jeu), **moyenne des différences appariées** sur les 15 couples (positif = meilleur que `ref16`) ; `± bruit` : écart-type de ces différences ;
- `mieux/15` : nombre de couples où le jeu bat `ref16` ;
- `p Wilcoxon` : indicatif seulement (voir section 3) ;
- `NASA moy` : score NASA moyen par prédiction ; `Δ NASA %` : variation vs `ref16` (positif = pire) ;
- `RMSE fin` : RMSE quand la RUL vraie ≤ 30.

In [ ]:
ref = resultats["ref16 : capteurs + time_cycles"]  # tableau de reference : 15 lignes (graine, pli)
lignes_tableau = []

for nom, config in configs.items():
    r = resultats[nom]
    # memes plis, dans le meme ordre -> la difference ligne a ligne est une difference appariee
    assert (r["graine"].values == ref["graine"].values).all() and (r["pli"].values == ref["pli"].values).all()
    gains = ref["rmse"].values - r["rmse"].values  # positif = ce jeu est meilleur que ref16 sur ce pli

    if nom.startswith("ref16"):
        p_wilcoxon = np.nan  # comparaison de ref16 avec elle-meme : sans objet
    else:
        p_wilcoxon = float(stats.wilcoxon(gains).pvalue)

    rmse_42 = r[r["graine"] == 42]["rmse"]  # les 5 plis du notebook 03
    lignes_tableau.append({
        "jeu": nom,
        "famille": config["famille"],
        "nb": len(config["colonnes"]),
        "RMSE 42": f"{rmse_42.mean():.2f} ± {rmse_42.std(ddof=0):.2f}",
        "RMSE 15": r["rmse"].mean(),
        "gain": gains.mean(),
        "± bruit": gains.std(ddof=0),
        "mieux/15": int(np.sum(gains > 0)),
        "p Wilcoxon": p_wilcoxon,
        "MAE": r["mae"].mean(),
        "NASA moy": r["nasa_moyen"].mean(),
        "Δ NASA %": 100 * (r["nasa_moyen"].mean() / ref["nasa_moyen"].mean() - 1),
        "RMSE fin": r["rmse_fin"].mean(),
    })

tableau = pd.DataFrame(lignes_tableau)

# regle de decision ecrite en section 3
tableau["retenu"] = (
    (tableau["gain"] >= 0.30) & (tableau["mieux/15"] >= 12) & (tableau["Δ NASA %"] <= 5.0)
    & (tableau["famille"] != "ref")
)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
affichage = tableau.copy()
for colonne in ["RMSE 15", "gain", "± bruit", "MAE", "RMSE fin"]:
    affichage[colonne] = affichage[colonne].map(lambda v: f"{v:.2f}")
affichage["p Wilcoxon"] = affichage["p Wilcoxon"].map(lambda v: "" if np.isnan(v) else f"{v:.3f}")
affichage["NASA moy"] = affichage["NASA moy"].map(lambda v: f"{v:.2f}")
affichage["Δ NASA %"] = affichage["Δ NASA %"].map(lambda v: f"{v:+.1f}")
print(affichage.drop(columns=["famille"]).to_string(index=False))

tableau.to_csv("../docs/figures/04e_tableau_ablation.csv", index=False)  # trace du tableau complet

In [ ]:
# ---- figure : gain de RMSE vs ref16, jeu par jeu, avec le bruit (ecart-type des differences appariees) ----
couleurs_famille = {"ref": "tab:grey", "F1": "tab:blue", "F2": "tab:orange", "F3": "tab:green", "F4": "tab:red", "F5": "tab:purple"}
noms_famille = {"ref": "référence", "F1": "F1 variance/skew/kurt", "F2": "F2 pente", "F3": "F3 fenêtres", "F4": "F4 EWMA", "F5": "F5 écart à l'état initial"}

a_tracer = tableau[~tableau["jeu"].str.startswith("ref16")].reset_index(drop=True)  # ref16 = ligne zero
positions = np.arange(len(a_tracer))

fig, ax = plt.subplots(figsize=(10, 11))
for i in range(len(a_tracer)):  # une barre par jeu
    ligne = a_tracer.iloc[i]
    ax.barh(positions[i], ligne["gain"], xerr=ligne["± bruit"], color=couleurs_famille[ligne["famille"]],
            alpha=0.85, ecolor="black", capsize=2)

ax.axvline(0, color="black", linewidth=1)
ax.axvline(0.30, color="grey", linestyle="--", label="seuil de la règle : gain ≥ 0,30 cycle")
ax.set_yticks(positions)
ax.set_yticklabels(a_tracer["jeu"], fontsize=8)
ax.invert_yaxis()  # premier jeu en haut
ax.set_xlabel("gain de RMSE vs ref16 (cycles, positif = meilleur) — barre d'erreur = ± 1 écart-type des différences appariées")
ax.set_title("Ablation : chaque jeu ajouté aux 16 features de référence (15 couples graine × pli)")

poignees = []
for cle, couleur in couleurs_famille.items():
    if cle != "ref":
        poignees.append(plt.Rectangle((0, 0), 1, 1, color=couleur, label=noms_famille[cle]))
poignees.append(plt.Rectangle((0, 0), 1, 1, color=couleurs_famille["ref"], label="ref46 (actuel)"))
legende_seuil = ax.get_legend_handles_labels()[0]
ax.legend(handles=poignees + legende_seuil, loc="upper right", fontsize=8)  # en haut a droite : les grandes barres F5 sont en bas

plt.tight_layout()
plt.savefig("../docs/figures/04e_ablation_gain_rmse.png", dpi=120)
plt.show()

## 6. Sélection (règle de la section 3)

On applique la règle telle qu'écrite. Pour chaque famille on garde, parmi ses variantes *retenues*, celle au meilleur gain. Les meilleures variantes retenues sont ensuite combinées en un seul jeu (si plusieurs familles passent), évalué comme les autres, puis le jeu final est choisi (meilleur RMSE, ou le plus simple à moins de 0,10 cycle).

In [ ]:
retenus = tableau[tableau["retenu"]]
print(f"Jeux qui passent la règle : {len(retenus)} sur {len(tableau) - 2} candidats (hors ref16 et ref46)\n")
if len(retenus) > 0:
    print(retenus[["jeu", "nb", "gain", "± bruit", "mieux/15", "Δ NASA %"]].to_string(index=False))

# meilleure variante retenue de chaque famille
meilleur_par_famille = {}
for famille in ["F1", "F2", "F3", "F4", "F5"]:
    candidats_famille = retenus[retenus["famille"] == famille]
    if len(candidats_famille) > 0:
        meilleur = candidats_famille.sort_values("gain", ascending=False).iloc[0]
        meilleur_par_famille[famille] = meilleur["jeu"]
print(f"\nMeilleure variante retenue par famille : {meilleur_par_famille}")

In [ ]:
# ---- jeux finalistes : ref16, chaque meilleure variante retenue, et leur combinaison ----
finalistes = {"ref16 : capteurs + time_cycles": configs["ref16 : capteurs + time_cycles"]["colonnes"]}
for famille, nom in meilleur_par_famille.items():
    finalistes[nom] = configs[nom]["colonnes"]

if len(meilleur_par_famille) >= 2:  # une combinaison n'a de sens qu'avec au moins deux familles
    colonnes_combinees = list(base16)  # on part des 16 de reference
    for famille, nom in meilleur_par_famille.items():
        for colonne in configs[nom]["colonnes"]:
            if colonne not in colonnes_combinees:  # sans doublon
                colonnes_combinees.append(colonne)
    nom_combinaison = "combinaison : " + " + ".join(meilleur_par_famille.values())
    configs[nom_combinaison] = {"famille": "combi", "colonnes": colonnes_combinees}
    resultats[nom_combinaison] = evaluer_jeu(colonnes_combinees)
    finalistes[nom_combinaison] = colonnes_combinees

# evaluation de chaque finaliste avec la MEME regle que les autres (gain pairs vs ref16)
lignes_finales = []
for nom, colonnes in finalistes.items():
    r = resultats[nom]
    gains = ref["rmse"].values - r["rmse"].values
    delta_nasa = 100 * (r["nasa_moyen"].mean() / ref["nasa_moyen"].mean() - 1)
    passe = (gains.mean() >= 0.30) and (np.sum(gains > 0) >= 12) and (delta_nasa <= 5.0) and (not nom.startswith("ref16"))
    lignes_finales.append({"jeu": nom, "nb": len(colonnes), "RMSE 15": r["rmse"].mean(), "gain": gains.mean(),
                           "± bruit": gains.std(ddof=0), "mieux/15": int(np.sum(gains > 0)),
                           "Δ NASA %": delta_nasa, "RMSE fin": r["rmse_fin"].mean(), "passe": passe})
finales = pd.DataFrame(lignes_finales)
print(finales.round(3).to_string(index=False))

# ---- choix final : meilleur RMSE parmi les jeux qui passent, sinon ref16 ; a moins de 0.10 du meilleur -> le plus simple ----
passants = finales[finales["passe"]]
if len(passants) == 0:
    nom_final = "ref16 : capteurs + time_cycles"
    print("\nAucun jeu ne passe la règle -> jeu final = ref16 (16 features).")
else:
    meilleur_gain = passants["gain"].max()
    proches = passants[passants["gain"] >= meilleur_gain - 0.10]  # a moins de 0.10 cycle du meilleur
    nom_final = proches.sort_values(["nb", "RMSE 15"]).iloc[0]["jeu"]  # le moins de features, puis le meilleur RMSE
    print(f"\nJeu final : {nom_final}")

colonnes_finales = list(finalistes[nom_final])
print(f"{len(colonnes_finales)} features retenues.")

## 7. Importance des features du jeu retenu : gain XGBoost **et** permutation

- **Gain XGBoost** : réduction totale de l'erreur apportée par les découpes qui utilisent la feature (modèle entraîné sur les 100 moteurs). Biaisé en faveur des features avec beaucoup de valeurs distinctes et instable entre features corrélées.
- **Importance par permutation** : on mélange au hasard les valeurs d'**une** colonne sur les moteurs de validation (jamais vus à l'entraînement) et on mesure la hausse de RMSE. Mesure ce que le modèle perd réellement sans cette information, sur des données non apprises. Calculée pli par pli (graine 42), 5 mélanges par pli.

Limite connue : quand deux features sont corrélées, mélanger l'une laisse l'autre compenser, donc leur importance par permutation est **sous-estimée** (surtout les capteurs lissés entre eux).

In [ ]:
X_final = X_train[colonnes_finales]

# ---- importance par gain : un modele entraine sur tous les moteurs ----
modele_gain = XGBRegressor(importance_type="gain", **REGLAGES_XGB)
modele_gain.fit(X_final, y)
importance_gain = pd.Series(modele_gain.feature_importances_, index=colonnes_finales)  # normalisee : somme = 1

# ---- importance par permutation : sur les moteurs de VALIDATION de chaque pli (graine 42) ----
hausses_par_pli = []
for idx_train, idx_val in plis_par_graine[42]:
    modele = XGBRegressor(**REGLAGES_XGB)
    modele.fit(X_final.iloc[idx_train], y.iloc[idx_train])
    resultat = permutation_importance(
        modele, X_final.iloc[idx_val], y.iloc[idx_val],
        scoring="neg_root_mean_squared_error",  # hausse de RMSE quand on melange la colonne
        n_repeats=5, random_state=42, n_jobs=1,
    )
    hausses_par_pli.append(resultat.importances_mean)  # un tableau (une valeur par colonne) par pli

hausses = np.array(hausses_par_pli)  # forme (5 plis, nb colonnes)
importance_permutation = pd.Series(hausses.mean(axis=0), index=colonnes_finales)
ecart_permutation = pd.Series(hausses.std(axis=0), index=colonnes_finales)  # variation entre plis

tableau_importance = pd.DataFrame({
    "gain (part)": importance_gain,
    "permutation (hausse RMSE)": importance_permutation,
    "± entre plis": ecart_permutation,
}).sort_values("permutation (hausse RMSE)", ascending=False)
print(tableau_importance.round(4).head(20).to_string())

In [ ]:
# ---- figure : les deux importances cote a cote, memes features, classees par permutation ----
top = tableau_importance.head(16).iloc[::-1]  # 16 premieres, la plus importante en haut du graphique
positions = np.arange(len(top))

fig, (ax_gain, ax_perm) = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
ax_gain.barh(positions, top["gain (part)"], color="tab:blue")
ax_gain.set_title("Importance par gain XGBoost (part du total)")
ax_gain.set_yticks(positions)
ax_gain.set_yticklabels(top.index, fontsize=8)

ax_perm.barh(positions, top["permutation (hausse RMSE)"], xerr=top["± entre plis"], color="tab:orange", ecolor="black", capsize=2)
ax_perm.set_title("Importance par permutation (hausse de RMSE, cycles)")
fig.suptitle(f"Jeu retenu ({len(colonnes_finales)} features) : les 16 features les plus importantes par permutation")

plt.tight_layout()
plt.savefig("../docs/figures/04e_importances_jeu_retenu.png", dpi=120)
plt.show()

# les deux classements s'accordent-ils ? (correlation de rang de Spearman entre les deux importances)
rho = stats.spearmanr(tableau_importance["gain (part)"], tableau_importance["permutation (hausse RMSE)"]).statistic
print(f"Correlation de rang gain / permutation (toutes les features) : {rho:.2f}")

## 8. Test officiel — utilisé **une seule fois**, pour le jeu retenu

Le jeu final est entraîné sur les 100 moteurs d'entraînement, puis évalué sur `test_FD001.txt` + `RUL_FD001.txt` : une prédiction par moteur (son dernier cycle observé), comparée à la vraie RUL **non plafonnée** (le test n'est pas la même mesure que la validation, protocole §1.5).

Pour la comparaison avec le XGBoost de la Phase 3, le modèle `ref46` est ré-entraîné et évalué de la même façon : ses chiffres (19,70 / 852,9 / 9 sur 100) sont déjà publiés, ce n'est donc pas une nouvelle lecture du test pour un choix ; on s'en sert pour une comparaison **moteur par moteur** (§8 du protocole).

In [ ]:
test_brut = load_fd001("test").drop(columns=capteurs_constants)  # memes capteurs retires que le train
test_norm = test_brut.copy()
test_norm[capteurs] = scaler.transform(test_brut[capteurs])  # transform seul : jamais de fit sur le test
ordre_test = test_norm.sort_values(["unit_number", "time_cycles"]).index
assert (ordre_test == test_norm.index).all(), "test non trie par moteur puis cycle"

X_test_tout = preparer_features(test_norm, capteurs)  # EXACTEMENT le meme code que pour le train

indices_derniers = test_norm.groupby("unit_number")["time_cycles"].idxmax()  # dernier cycle observe de chaque moteur
derniers = X_test_tout.loc[indices_derniers.sort_index().values]
rul_vraie_test = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])["RUL_vraie"].values
assert len(derniers) == 100 and len(rul_vraie_test) == 100


def evaluer_sur_test(colonnes):
    """Entraine sur les 100 moteurs du train avec ces colonnes, predit le dernier cycle des 100 moteurs de test."""
    modele = XGBRegressor(**REGLAGES_XGB)
    modele.fit(X_train[colonnes], y)
    prediction = np.clip(modele.predict(derniers[colonnes]), 0, None)  # une RUL negative n'a pas de sens
    erreur = prediction - rul_vraie_test
    return prediction, {
        "RMSE": float(np.sqrt(np.mean(erreur ** 2))),
        "MAE": float(np.mean(np.abs(erreur))),
        "score NASA (somme)": score_nasa_boucle(rul_vraie_test, prediction),
        "score NASA moyen": score_nasa_moyen(rul_vraie_test, prediction),
        "surestimes > 20": int(np.sum(erreur > 20)),
    }


colonnes_ref46 = configs["ref46 : + moy_5 + std_5 (actuel)"]["colonnes"]
pred_ref46, metriques_ref46 = evaluer_sur_test(colonnes_ref46)
pred_final, metriques_final = evaluer_sur_test(colonnes_finales)

comparaison = pd.DataFrame({"ref46 (Phase 3)": metriques_ref46, f"retenu ({len(colonnes_finales)} features)": metriques_final})
print(comparaison.round(2).to_string())
if abs(metriques_ref46["RMSE"] - 19.70) > 0.15:
    print("ATTENTION : ref46 ne redonne pas le 19.70 publie ; verifier l'environnement (variation XGBoost)")

# comparaison moteur par moteur (erreurs absolues), test de Wilcoxon apparie
erreur_abs_ref46 = np.abs(pred_ref46 - rul_vraie_test)
erreur_abs_final = np.abs(pred_final - rul_vraie_test)
if np.allclose(erreur_abs_ref46, erreur_abs_final):
    print("\nModeles identiques sur le test : pas de comparaison appariee.")
else:
    p = stats.wilcoxon(erreur_abs_ref46 - erreur_abs_final).pvalue
    moteurs_mieux = int(np.sum(erreur_abs_final < erreur_abs_ref46))
    print(f"\nErreur absolue moteur par moteur : le jeu retenu est meilleur sur {moteurs_mieux}/100 moteurs, Wilcoxon apparie p = {p:.3f}")

In [ ]:
# ---- synthese chiffree (a recopier dans le rapport) ----
final_ligne = finales[finales["jeu"] == nom_final].iloc[0]
ref46_ligne = tableau[tableau["jeu"].str.startswith("ref46")].iloc[0]
ref16_ligne = tableau[tableau["jeu"].str.startswith("ref16")].iloc[0]
print(f"Jeu retenu : {nom_final} ({len(colonnes_finales)} features)")
print(f"  validation (15 couples graine x pli) : RMSE {final_ligne['RMSE 15']:.2f}   | ref16 {ref16_ligne['RMSE 15']:.2f}   | ref46 {ref46_ligne['RMSE 15']:.2f}")
print(f"  gain apparie vs ref16 : {final_ligne['gain']:+.2f} +/- {final_ligne['± bruit']:.2f} (bat ref16 sur {final_ligne['mieux/15']}/15)")
print(f"  gain apparie ref46 vs ref16 : {ref46_ligne['gain']:+.2f} +/- {ref46_ligne['± bruit']:.2f}")
print(f"  test officiel : RMSE {metriques_final['RMSE']:.2f}, score NASA {metriques_final['score NASA (somme)']:.1f}, surestimes>20 {metriques_final['surestimes > 20']}/100")

print("\nJeux ecartes (aucun ne passe la regle) ou non retenus : voir le tableau section 5, colonne 'retenu'.")

## 9. Extension, décidée **après** avoir lu les résultats des sections 5 à 8

> Cette section n'était pas prévue au départ. Elle répond à deux constats faits en lisant le tableau d'ablation, et **ne modifie ni la sélection (section 6) ni le test officiel (section 8)** : le jeu retenu reste `ecartinit_20`, et aucune de ces variantes n'est évaluée sur le test.

1. **Effet de bord de la grille.** Pour `ecartinit`, `pente` et `std`, le gain augmente avec la taille de fenêtre et la meilleure variante est **la plus grande fenêtre testée** (20 ou 30) : l'optimum n'est donc pas encadré. On teste des fenêtres plus longues (30, 50, 80 pour `ecartinit` ; 50 pour `pente` et `std`).
2. **Règle 9 (version plus simple à performance égale).** Le jeu retenu a 31 features alors que `time_cycles` et `ecartinit_20` semblent porter l'essentiel de l'importance. On teste des versions plus simples : 16 features (`ecartinit_20` + `time_cycles`, sans les capteurs lissés) et 30 features (sans `time_cycles`), plus une version à 31 features avec la pente.

Même validation (3 graines × 5 plis), mêmes réglages. `gain vs final` = RMSE(jeu retenu) − RMSE(variante), positif = la variante fait mieux que le jeu retenu.

In [ ]:
def ajouter_statistique(statistique, k):
    """Calcule la statistique (k cycles) pour les 15 capteurs lisses du train, moteur par moteur, et l'ajoute a X_train."""
    unites = train_norm["unit_number"].values
    nouvelles = {}
    for capteur in capteurs:
        nouvelles[f"{capteur}|{statistique}_{k}"] = np.zeros(len(X_train))  # espace reserve
    for moteur_id in np.unique(unites):
        positions = np.where(unites == moteur_id)[0]
        for capteur in capteurs:
            serie_lisse = pd.Series(X_train[capteur].values[positions])  # X_train contient deja les capteurs lisses
            valeurs = statistique_fenetre(serie_lisse, serie_lisse, statistique, k)  # 2e argument inutile hors EWMA
            nouvelles[f"{capteur}|{statistique}_{k}"][positions] = valeurs
    for nom_colonne, valeurs in nouvelles.items():
        X_train[nom_colonne] = valeurs


debut = time.time()
for k in [30, 50, 80]:
    ajouter_statistique("ecartinit", k)
for statistique in ["pente", "std"]:
    ajouter_statistique(statistique, 50)
print(f"Colonnes supplementaires calculees en {time.time() - debut:.0f} s ; X_train a maintenant {X_train.shape[1]} colonnes")

# les variantes de l'extension (aucune n'est evaluee sur le test)
extension = {}
for k in [30, 50, 80]:
    extension[f"ext F5 +ecartinit_{k}"] = base16 + colonnes_de("ecartinit", k)
extension["ext F2 +pente_50"] = base16 + colonnes_de("pente", 50)
extension["ext F3 +std_50"] = base16 + colonnes_de("std", 50)
extension["simplif 16 : ecartinit_20 + time_cycles"] = colonnes_de("ecartinit", 20) + ["time_cycles"]
extension["simplif 30 : capteurs + ecartinit_20 (sans time_cycles)"] = capteurs + colonnes_de("ecartinit", 20)
extension["simplif 31 : ecartinit_20 + time_cycles + pente_30"] = colonnes_de("ecartinit", 20) + ["time_cycles"] + colonnes_de("pente", 30)

resultats_final = resultats[nom_final]  # jeu retenu, pour la comparaison appariee
lignes_ext = []
for nom, colonnes in extension.items():
    r = evaluer_jeu(colonnes)
    resultats[nom] = r
    gain_ref16 = ref["rmse"].values - r["rmse"].values
    gain_final = resultats_final["rmse"].values - r["rmse"].values
    lignes_ext.append({
        "jeu": nom, "nb": len(colonnes), "RMSE 15": r["rmse"].mean(),
        "gain vs ref16": gain_ref16.mean(),
        "gain vs final": gain_final.mean(), "± bruit": gain_final.std(ddof=0), "mieux que final /15": int(np.sum(gain_final > 0)),
        "NASA moy": r["nasa_moyen"].mean(), "RMSE fin": r["rmse_fin"].mean(),
    })
    print(f"{nom:60s} RMSE = {r['rmse'].mean():.2f}")

tableau_ext = pd.DataFrame(lignes_ext)
print(f"\nPour memoire, jeu retenu ({nom_final}) : RMSE {resultats_final['rmse'].mean():.2f}, {len(colonnes_finales)} features")
print(tableau_ext.round(2).to_string(index=False))

## 10. Conclusions (chiffres lus dans les sorties ci-dessus ; deux exécutions complètes ont donné les mêmes valeurs)

> Rédigées **après** exécution. Les hypothèses de la section 4 n'ont pas été modifiées.

### Verdict par famille (gain de RMSE vs `ref16`, moyenne des différences appariées sur 15 couples graine × pli ; bruit typique entre plis ≈ 1,2)

| Famille | Hypothèse (section 4) | Résultat | Verdict |
|---|---|---|---|
| **F5** écart à l'état initial | gain plausible, « deuxième après la pente » | `ecartinit_5` **+6,07**, `_10` **+6,97**, `_20` **+7,38** (15/15 plis) ; `_30` et `_50` +7,59 ; `_80` +7,26 | **Confirmée, bien au-delà de l'attendu : de loin la meilleure famille** |
| **F2** pente | famille la plus plausible d'un gain, k=5 trop bruité | `pente_5` +0,39, `_10` +1,05, `_20` +3,30, `_30` +5,12, `_50` +7,30 | Confirmée ; le gain croît avec la fenêtre |
| **F3** fenêtres | moyennes redondantes ; écart-type long = léger gain | `moy_5` −0,04, `moy_10` +0,05 (redondantes, comme prévu) mais `moy_20` +0,86, `moy_30` +1,66 ; `std_10` +0,27, `std_20` +1,70, `std_30` +3,56, `std_50` +5,84 | Partiellement **infirmée** : les moyennes longues apportent (une moyenne sur 30 cycles est un niveau *passé*, que le niveau actuel peut comparer) ; le gain des écarts-types longs est bien plus grand que « léger » |
| **F1** variance / asymétrie / kurtosis | rien en k=5 ; au mieux un petit effet en k=20 | k=5 : `var` +0,10, `skew` +0,02, `kurt` +0,04. k=20 : `var` **+1,70**, `skew` +0,05, `kurt` +0,18 | Confirmée pour skew/kurt (rien). La variance à k=20 apporte, mais voir la remarque ci-dessous |
| **F4** EWMA | ajout redondant ; en remplacement du lissage, égalité | ajout : `ewma_5` +0,12, `_10` +0,44 (11/15), `_20` +1,18 (13/15, bruit 1,0) ; remplacement : +0,10, +0,12, −0,28 | Confirmée : en remplacement, jeu égal (différences de −0,28 à +0,12, dans le bruit) ; un autre filtre, pas une autre information |

**Remarque sur F1 et F3 :** `var_20` et `std_20` donnent exactement le même résultat (gain +1,70 pour les deux). C'est normal : la variance est le carré de l'écart-type, et les arbres de décision ne dépendent que de l'ordre des valeurs, pas de leur échelle. Ce ne sont pas deux informations, mais une seule.

### Jeu retenu (règle de la section 3, appliquée telle qu'écrite)

**`ref16` + `ecartinit_20`, soit 31 features** (15 capteurs lissés, `time_cycles`, 15 écarts à l'état initial sur 20 cycles).

| | `ref16` (16) | `ref46` (46, Phase 3) | **retenu (31)** |
|---|---|---|---|
| Validation, RMSE (15 couples) | 16,65 | 16,60 | **9,27** |
| Gain apparié vs `ref16` | — | +0,05 ± 0,17 | **+7,38 ± 1,37, 15/15** |
| Score NASA moyen par prédiction | 6,88 | | 1,47 (−78,6 %) |
| RMSE en fin de vie (RUL ≤ 30) | 9,41 | 9,48 | 3,69 |
| **Test officiel : RMSE** | | 19,70 | **13,10** |
| Test officiel : score NASA (somme) | | 852,9 | **237,0** |
| Test officiel : surestimés de plus de 20 cycles | | 9/100 | **2/100** |

Sur le test, le jeu retenu a une erreur absolue plus petite sur 60 moteurs sur 100, et le Wilcoxon apparié donne p < 0,001. Le benchmark exploratoire du README (RMSE test 18,97) est battu. **Le test n'a servi qu'une fois pour ce jeu** (plus une réévaluation de `ref46`, dont le chiffre était déjà publié, pour la comparaison moteur par moteur).

### Pourquoi 31 et pas moins ni plus (règle 9)

- **Combiner toutes les familles retenues (121 features) fait moins bien (9,58) que `ecartinit_20` seul (9,27)** : les familles se recouvrent, empiler ne sert à rien.
- **Plus simple, à performance égale ?** Non. `ecartinit_20` + `time_cycles` seuls (16 features) : 10,18 (−0,91 vs retenu, moins bon sur 15/15 plis). Sans `time_cycles` (30 features) : 12,42. Ajouter la pente à la version à 16 : 10,33. Le jeu à 31 features reste donc le plus simple des jeux au niveau de 9,3.
- **Fenêtre de `ecartinit`** : 30 et 50 font +0,21 de mieux que 20, dans le bruit (écart-type de la différence 0,23 à 0,31, et sous le seuil de 0,30 de la règle) ; 80 fait −0,12. L'optimum est donc autour de 20-50 et **bien encadré** par l'extension de la section 9. 20 est conservé : la règle ne justifie pas de changer.

### Écarté, avec la raison chiffrée

- `skew_5`, `skew_20`, `kurt_5`, `kurt_20`, `var_5`, `std_5`, `moy_5`, `moy_10`, `ewma_5`, `std_10` : gain de −0,04 à +0,27, sous le seuil de 0,30 et dans le bruit.
- Les colonnes de `ref46` (`moy_5` + `std_5`, 30 features) : **+0,05 ± 0,17**, indiscernable de zéro, 8 plis sur 15 seulement. Elles ne sont pas conservées.
- Remplacer le lissage par une EWMA : égalité (−0,28 à +0,12), aucun intérêt à changer un choix de la Phase 1 qui marche.
- La combinaison de toutes les familles (121 features) : 9,58, pire que `ecartinit_20` seul.

### Importance des features du jeu retenu (section 7)

- **Permutation** (hausse du RMSE quand on mélange la colonne, sur des moteurs jamais vus) : `time_cycles` 7,28 ; `sensor_11|ecartinit_20` 5,27 ; `sensor_9|ecartinit_20` 4,52 ; `sensor_4|ecartinit_20` 1,93 ; les capteurs lissés seuls pèsent au plus 0,86 (`sensor_4`).
- **Gain XGBoost** : `sensor_11|ecartinit_20` 0,38 ; `sensor_15|ecartinit_20` 0,14 ; `sensor_4|ecartinit_20` 0,11 ; `time_cycles` seulement 0,005. **Les deux mesures ne s'accordent pas sur `time_cycles`** (corrélation de rang globale 0,72). Le gain compte les découpes, pas ce que le modèle perd sans la variable ; et `time_cycles` reste décisif quand on l'enlève (12,42 au lieu de 9,27). On se fie à la permutation, avec sa limite connue : elle sous-estime les variables corrélées entre elles.

### Limites et points à vérifier (règle 11)

1. **`ecartinit` suppose qu'on observe le moteur depuis son cycle 1.** C'est vrai dans C-MAPSS (train et test commencent au cycle 1), pas forcément en exploitation : un moteur déployé en cours de vie n'a pas d'« état initial » connu. À écrire dans le rapport de phase.
2. **Le mécanisme n'est pas prouvé.** L'hypothèse (les moteurs partent de niveaux différents, et soustraire ce niveau isole la dégradation) est cohérente avec les résultats mais je n'ai pas mesuré la dispersion du niveau initial entre moteurs. Une figure « niveau initial de `sensor_11` pour les 100 moteurs » la confirmerait ou non.
3. **Optimisme de sélection.** La fenêtre (20) et la famille ont été choisies sur la validation ; le test indépendant (19,70 → 13,10) confirme le gain, mais l'ordre de grandeur de la validation (9,27) est, comme toujours, un peu optimiste.
4. **Les p-valeurs de Wilcoxon sur 15 couples sont optimistes** (les 3 graines réutilisent les mêmes 100 moteurs). Le critère « bat `ref16` sur 15 plis sur 15 » et la taille du gain retenu (+7,38, plus de 5 fois l'écart-type des différences appariées) pèsent plus que ces p-valeurs.
5. **Une affirmation faite plus tôt dans la session était à nuancer** : mesurer l'erreur « au dernier cycle de chaque moteur de validation » n'aurait rien apporté, puisque tous les moteurs d'entraînement finissent à RUL = 0. Évaluer toutes les lignes équivaut à moyenner sur des coupes de trajectoire uniformes ; ce qui diffère du test est la répartition des RUL (non plafonnées, de 7 à 145), pas le fait d'évaluer plusieurs cycles.
6. **Non fait ici** : normalisation du scaler à l'intérieur des plis (sans effet sur XGBoost, à faire pour un LSTM), FD002 à FD004 (normalisation par régime), réglage des hyperparamètres sur le nouveau jeu de features (autre session), importance SHAP (Phase 5). Les réglages XGBoost sont ceux de la Phase 3 ; les hyperparamètres optimaux ont pu changer avec ces features.